# 02 — Model Improvement Experiments

Notebook ini mencoba beberapa cara untuk memperbaiki evaluasi model: mengubah target menjadi binary, memakai stratified cross-validation, membandingkan beberapa model, memakai class weight, mencoba oversampling, dan mengecek data leakage.

Tujuan utamanya bukan memaksa skor tinggi, tetapi mencari pendekatan modeling yang paling masuk akal untuk dataset kecil.

## 1. Import Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, f1_score, balanced_accuracy_score

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE

## 2. Load dan Cleaning Dataset

Cleaning dilakukan supaya format data lebih konsisten sebelum masuk ke proses modeling.

In [2]:
DATA_PATH = Path("../data/Student Mental health.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/Student Mental health.csv")

raw_df = pd.read_csv(DATA_PATH)

df = raw_df.rename(columns={
    "Timestamp": "timestamp",
    "Choose your gender": "gender",
    "Age": "age",
    "What is your course?": "course",
    "Your current year of Study": "year_of_study",
    "What is your CGPA?": "cgpa",
    "Marital status": "marital_status",
    "Do you have Depression?": "depression",
    "Do you have Anxiety?": "anxiety",
    "Do you have Panic attack?": "panic_attack",
    "Did you seek any specialist for a treatment?": "seek_treatment",
})

df["gender"] = df["gender"].fillna("Unknown").astype(str).str.strip().str.title()
df["course"] = df["course"].fillna("Unknown").astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())
df["year_of_study"] = (
    df["year_of_study"].astype(str).str.extract(r"(\d+)")[0]
    .fillna("0").astype(int).map(lambda x: f"Year {x}" if x else "Unknown")
)
df["cgpa"] = df["cgpa"].fillna("Unknown").astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

for col in ["depression", "anxiety", "panic_attack", "seek_treatment"]:
    df[col] = np.where(df[col].astype(str).str.strip().str.title() == "Yes", "Yes", "No")

df.head()

,timestamp,gender,age,course,year_of_study,cgpa,marital_status,depression,anxiety,panic_attack,seek_treatment
0,8/7/2020 12:02,Female,18.0,Engineering,Year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic Education,Year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,Bit,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,Year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,Year 4,3.00 - 3.49,No,No,No,No,No


## 3. Membuat Target Multiclass dan Binary

Target multiclass tetap menggunakan `Low Risk`, `Medium Risk`, dan `High Risk`. Target binary menggabungkan `Medium Risk` dan `High Risk` menjadi `At Risk`, sehingga lebih cocok untuk screening awal.

In [3]:
df["symptom_count"] = (
    (df["depression"] == "Yes").astype(int)
    + (df["anxiety"] == "Yes").astype(int)
    + (df["panic_attack"] == "Yes").astype(int)
)

df["risk_level"] = df["symptom_count"].map(
    lambda x: "Low Risk" if x == 0 else ("Medium Risk" if x == 1 else "High Risk")
)
df["at_risk"] = np.where(df["symptom_count"] > 0, "At Risk", "No Risk")

print("Jumlah data:", df.shape[0])
print("\nDistribusi risk_level:")
print(df["risk_level"].value_counts())
print("\nDistribusi target binary:")
print(df["at_risk"].value_counts())

Jumlah data: 101

Distribusi risk_level:
risk_level
Low Risk       37
Medium Risk    36
High Risk      28
Name: count, dtype: int64

Distribusi target binary:
at_risk
At Risk    64
No Risk    37
Name: count, dtype: int64


## 4. Preprocessing, Fitur, dan Model

Fitur `depression`, `anxiety`, dan `panic_attack` tidak dipakai dalam model valid karena label risiko dibuat dari tiga indikator tersebut. Kalau dipakai, hasilnya akan tinggi tetapi bocor.

In [4]:
FEATURES = ["age", "gender", "course", "year_of_study", "cgpa", "marital_status"]
NUM_FEATURES = ["age"]
CAT_FEATURES = ["gender", "course", "year_of_study", "cgpa", "marital_status"]

X = df[FEATURES]
y_multi = df["risk_level"]
y_binary = df["at_risk"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), NUM_FEATURES),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), CAT_FEATURES),
])

models = {
    "Dummy Most Frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Logistic Regression Balanced": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree Balanced": DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, class_weight="balanced", random_state=42),
    "Random Forest Balanced": RandomForestClassifier(n_estimators=40, max_depth=4, min_samples_leaf=3, class_weight="balanced", random_state=42),
    "Extra Trees Balanced": ExtraTreesClassifier(n_estimators=40, max_depth=4, min_samples_leaf=3, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

## 5. Fungsi Evaluasi Cross-Validation

In [5]:
def evaluate_models(X, y, experiment_name, model_dict):
    rows = []
    for model_name, model in model_dict.items():
        pipe = Pipeline([
            ("preprocess", preprocessor),
            ("model", model)
        ])
        scores = cross_validate(pipe, X, y, scoring=scoring, cv=cv, n_jobs=1)
        rows.append({
            "experiment": experiment_name,
            "model": model_name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
            "f1_macro_mean": scores["test_f1_macro"].mean(),
            "f1_weighted_mean": scores["test_f1_weighted"].mean(),
        })
    return pd.DataFrame(rows).sort_values("f1_macro_mean", ascending=False)

## 6. Eksperimen 1 — Multiclass Risk Level

In [6]:
result_multi = evaluate_models(X, y_multi, "Multiclass Risk Level", models)
result_multi.round(3)

,experiment,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean,f1_weighted_mean
5,Multiclass Risk Level,Extra Trees Balanced,0.515,0.517,0.516,0.507
6,Multiclass Risk Level,Gradient Boosting,0.445,0.448,0.442,0.440
2,Multiclass Risk Level,Logistic Regression Balanced,0.436,0.433,0.423,0.426
1,Multiclass Risk Level,Logistic Regression,0.436,0.432,0.422,0.424
4,Multiclass Risk Level,Random Forest Balanced,0.406,0.404,0.386,0.385
3,Multiclass Risk Level,Decision Tree Balanced,0.346,0.348,0.279,0.278
0,Multiclass Risk Level,Dummy Most Frequent,0.366,0.333,0.179,0.197


## 7. Eksperimen 2 — Binary At Risk

Hasil binary biasanya lebih stabil karena tugas model lebih sederhana: membedakan mahasiswa yang tidak punya indikator dan yang memiliki minimal satu indikator.

In [7]:
result_binary = evaluate_models(X, y_binary, "Binary At Risk", models)
result_binary.round(3)

,experiment,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean,f1_weighted_mean
6,Binary At Risk,Gradient Boosting,0.702,0.633,0.630,0.674
1,Binary At Risk,Logistic Regression,0.683,0.600,0.576,0.634
2,Binary At Risk,Logistic Regression Balanced,0.593,0.585,0.574,0.595
3,Binary At Risk,Decision Tree Balanced,0.573,0.610,0.570,0.572
5,Binary At Risk,Extra Trees Balanced,0.564,0.607,0.548,0.549
4,Binary At Risk,Random Forest Balanced,0.534,0.539,0.502,0.520
0,Binary At Risk,Dummy Most Frequent,0.634,0.500,0.388,0.492


## 8. Eksperimen 3 — Oversampling untuk Binary Target

Oversampling dicoba hanya untuk target binary agar eksperimen tetap ringan dan relevan dengan sistem early warning.

In [8]:
oversampling_models = {
    "LogReg Balanced + ROS": ImbPipeline([
        ("preprocess", preprocessor),
        ("oversample", RandomOverSampler(random_state=42)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "LogReg Balanced + SMOTE": ImbPipeline([
        ("preprocess", preprocessor),
        ("oversample", SMOTE(random_state=42, k_neighbors=3)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "RF + ROS": ImbPipeline([
        ("preprocess", preprocessor),
        ("oversample", RandomOverSampler(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=40, max_depth=4, min_samples_leaf=3, random_state=42)),
    ]),
}

result_oversampling = []
for model_name, pipe in oversampling_models.items():
    scores = cross_validate(pipe, X, y_binary, scoring=scoring, cv=cv, n_jobs=1)
    result_oversampling.append({
        "experiment": "Binary At Risk with Oversampling",
        "model": model_name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_weighted_mean": scores["test_f1_weighted"].mean(),
    })

result_oversampling = pd.DataFrame(result_oversampling).sort_values("f1_macro_mean", ascending=False)
result_oversampling.round(3)

,experiment,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean,f1_weighted_mean
0,Binary At Risk with Oversampling,LogReg Balanced + ROS,0.642,0.613,0.612,0.639
2,Binary At Risk with Oversampling,RF + ROS,0.613,0.607,0.596,0.615
1,Binary At Risk with Oversampling,LogReg Balanced + SMOTE,0.604,0.577,0.570,0.600


## 9. Leakage Demo — Skor Tinggi tapi Tidak Valid

Bagian ini sengaja memasukkan `depression`, `anxiety`, dan `panic_attack` sebagai fitur untuk memprediksi `risk_level`. Karena `risk_level` dibuat dari tiga indikator itu, hasilnya sangat tinggi, tetapi tidak valid untuk diklaim sebagai model prediksi.

In [9]:
LEAK_FEATURES = FEATURES + ["depression", "anxiety", "panic_attack"]
X_leak = df[LEAK_FEATURES]

leak_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), NUM_FEATURES),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), CAT_FEATURES + ["depression", "anxiety", "panic_attack"]),
])

leak_models = {
    "LogReg Leakage": LogisticRegression(max_iter=1000, random_state=42),
    "RF Leakage": RandomForestClassifier(n_estimators=40, max_depth=4, min_samples_leaf=3, random_state=42),
}

leak_rows = []
for model_name, model in leak_models.items():
    pipe = Pipeline([
        ("preprocess", leak_preprocessor),
        ("model", model)
    ])
    scores = cross_validate(pipe, X_leak, y_multi, scoring=scoring, cv=cv, n_jobs=1)
    leak_rows.append({
        "experiment": "Leakage Demo - Invalid",
        "model": model_name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_weighted_mean": scores["test_f1_weighted"].mean(),
    })

result_leakage = pd.DataFrame(leak_rows).sort_values("f1_macro_mean", ascending=False)
result_leakage.round(3)

,experiment,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean,f1_weighted_mean
1,Leakage Demo - Invalid,RF Leakage,0.96,0.958,0.958,0.959
0,Leakage Demo - Invalid,LogReg Leakage,0.93,0.924,0.925,0.929


## 10. Holdout Report Model Terbaik

Holdout split tetap ditampilkan sebagai gambaran tambahan, tetapi kesimpulan utama tetap memakai cross-validation karena dataset kecil.

In [10]:
def holdout_report(X, y, model, title):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(title)
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
    print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred), 3))
    print("F1 Macro:", round(f1_score(y_test, y_pred, average="macro"), 3))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=3))

best_multi_model_name = result_multi.iloc[0]["model"]
best_binary_model_name = result_binary.iloc[0]["model"]

print("Best multiclass model from CV:", best_multi_model_name)
print("Best binary model from CV:", best_binary_model_name)
print("-" * 70)
holdout_report(X, y_multi, models[best_multi_model_name], f"Multiclass Holdout — {best_multi_model_name}")
print("-" * 70)
holdout_report(X, y_binary, models[best_binary_model_name], f"Binary Holdout — {best_binary_model_name}")

Best multiclass model from CV: Extra Trees Balanced
Best binary model from CV: Gradient Boosting
----------------------------------------------------------------------
Multiclass Holdout — Extra Trees Balanced
Accuracy: 0.5
Balanced Accuracy: 0.48
F1 Macro: 0.444

Classification Report:
              precision    recall  f1-score   support

   High Risk      0.750     0.429     0.545         7
    Low Risk      0.474     0.900     0.621        10
 Medium Risk      0.333     0.111     0.167         9

    accuracy                          0.500        26
   macro avg      0.519     0.480     0.444        26
weighted avg      0.499     0.500     0.443        26

----------------------------------------------------------------------
Binary Holdout — Gradient Boosting
Accuracy: 0.538
Balanced Accuracy: 0.512
F1 Macro: 0.512

Classification Report:
              precision    recall  f1-score   support

     At Risk      0.625     0.625     0.625        16
     No Risk      0.400     0.400  

## 11. Ringkasan Akhir Eksperimen Valid

In [11]:
all_valid_results = pd.concat([result_multi, result_binary, result_oversampling], ignore_index=True)
all_valid_results_sorted = all_valid_results.sort_values("f1_macro_mean", ascending=False)
all_valid_results_sorted.round(3)

,experiment,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean,f1_weighted_mean
7,Binary At Risk,Gradient Boosting,0.702,0.633,0.630,0.674
14,Binary At Risk with Oversampling,LogReg Balanced + ROS,0.642,0.613,0.612,0.639
15,Binary At Risk with Oversampling,RF + ROS,0.613,0.607,0.596,0.615
8,Binary At Risk,Logistic Regression,0.683,0.600,0.576,0.634
9,Binary At Risk,Logistic Regression Balanced,0.593,0.585,0.574,0.595
16,Binary At Risk with Oversampling,LogReg Balanced + SMOTE,0.604,0.577,0.570,0.600
10,Binary At Risk,Decision Tree Balanced,0.573,0.610,0.570,0.572
11,Binary At Risk,Extra Trees Balanced,0.564,0.607,0.548,0.549
0,Multiclass Risk Level,Extra Trees Balanced,0.515,0.517,0.516,0.507
12,Binary At Risk,Random Forest Balanced,0.534,0.539,0.502,0.520


## 12. Kesimpulan

1. Model 3 kelas masih terbatas karena dataset kecil dan fitur yang tersedia belum cukup kuat untuk membedakan `Low`, `Medium`, dan `High Risk`.
2. Target binary `No Risk` vs `At Risk` lebih cocok untuk early warning awal.
3. Oversampling bisa membantu pada beberapa model, tetapi tidak selalu lebih baik.
4. Skor leakage sangat tinggi, tetapi tidak valid karena memakai indikator yang membentuk target.
5. Untuk lomba, model sebaiknya dijadikan proof of concept. Kekuatan solusi tetap ada pada risk scoring, dashboard, treatment gap analysis, dan desain decision support system.

In [12]:
OUTPUT_DIR = Path("../outputs")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

all_valid_results_sorted.to_csv(OUTPUT_DIR / "model_improvement_experiments.csv", index=False)
result_leakage.to_csv(OUTPUT_DIR / "leakage_demo_results.csv", index=False)

print("Saved:")
print("-", OUTPUT_DIR / "model_improvement_experiments.csv")
print("-", OUTPUT_DIR / "leakage_demo_results.csv")

Saved:
- ../outputs/model_improvement_experiments.csv
- ../outputs/leakage_demo_results.csv
